# EPCLM-004 M2 — Baseline Architectures
T4 + Internet + existing GITHUB_TOKEN, Save & Run All. Frozen E0, E1 SWA, pinned Mamba-1 reference E2, Samba-lite E3. Common data/8192 tokenizer/FP32/2 updates per arm. No M1 remedial training, no 20M/100M. PyTorch reference costs are not native-kernel costs.

In [ ]:
from pathlib import Path
import os, signal, subprocess, sys, time
started = time.monotonic()
hard_deadline = started + 7200
repo = Path('/kaggle/working/minicells-epclm004-m2')
branch = 'research/epclm-004-m2-baselines'

def run(command, cwd=None, limit=180, stop_at=None):
    remaining = min(limit, (stop_at or hard_deadline)-time.monotonic())
    if remaining <= 0: raise TimeoutError('M2 budget exhausted; Output retained')
    p = subprocess.Popen(command, cwd=cwd, start_new_session=True)
    try: code = p.wait(timeout=remaining)
    except subprocess.TimeoutExpired:
        os.killpg(p.pid, signal.SIGTERM)
        try: p.wait(timeout=15)
        except subprocess.TimeoutExpired:
            os.killpg(p.pid,signal.SIGKILL);p.wait()
        raise TimeoutError('M2 stopped at budget boundary; inspect Kaggle Output')
    if code: raise RuntimeError('M2 stage failed; inspect decision.json in Kaggle Output')

if not repo.exists():
    run(['git','clone','--depth=1','--single-branch','--branch',branch,'https://github.com/ArcheLabs/mini-cells.git',str(repo)])
else:
    origin = subprocess.check_output(['git','-C',str(repo),'remote','get-url','origin'],text=True).strip()
    if origin.removesuffix('.git') != 'https://github.com/ArcheLabs/mini-cells': raise ValueError('unrelated checkout')
    run(['git','-C',str(repo),'fetch','--depth=1','origin',branch])
    run(['git','-C',str(repo),'switch','--detach','FETCH_HEAD'])
source = subprocess.check_output(['git','-C',str(repo),'rev-parse','HEAD'],text=True).strip()
base = repo/'research/projects/cellular-language-models/experiments/epclm-004-unified-memory-10m/m2'
run([sys.executable,'-m','pip','install','-q','-r',str(base/'requirements.txt')],limit=300)
os.environ['OMP_NUM_THREADS']='1'
os.environ['PYTEST_DISABLE_PLUGIN_AUTOLOAD']='1'
run([sys.executable,'-m','pytest','-q',str(base/'tests')],cwd=repo,limit=180)
# No M1 remedial training. All four arms get the same two-update M2 correctness
# smoke from fresh shared initialization; no quality ranking is authorized.
run([sys.executable,str(base/'run.py'),'--device','cuda',
     '--assets','/kaggle/working/epclm004-m1-assets',
     '--output',f'/kaggle/working/epclm004-m2-{source[:12]}',
     '--started',str(started),'--publish'],cwd=repo,limit=6000,stop_at=started+6000)
